In [ ]:
! pip install bs4 # in case you don't have it installed
! pip install contractions
! pip install scikit-learn

# Dataset: https://s3.amazonaws.com/amazon-reviews-pds/tsv/amazon_reviews_us_Beauty_v1_00.tsv.gz
#          https://web.archive.org/web/20201127142707if_/https://s3.amazonaws.com/amazon-reviews-pds/tsv/amazon_reviews_us_Office_Products_v1_00.tsv.gz

In [2]:
import pandas as pd
import numpy as np
import nltk
import csv
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('stopwords')
import re
from bs4 import BeautifulSoup
import contractions
import warnings
warnings.filterwarnings("ignore")

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Dataset Preparation

## Read Data

In [3]:
raw_data = pd.read_csv('./data.tsv', sep='\t', quoting=csv.QUOTE_NONE)

## Keep Reviews and Ratings

*Sample reviews and ratings*

In [4]:
data = raw_data.loc[:, ['review_body', 'star_rating']]
data.sample(3, random_state=42)

,review_body,star_rating
267839,GREAT!!!!,5
531637,Awfully expensive for a cardboard box and asse...,3
1171617,What a great deal all items were packaged well...,5


*Rating statistics*

In [5]:
data.describe()

,star_rating
count,2.642434e+06
mean,4.072539e+00
std,1.386968e+00
min,1.000000e+00
25%,4.000000e+00
50%,5.000000e+00
75%,5.000000e+00
max,5.000000e+00


*Count of each type of rating*

In [6]:
data['star_rating'].value_counts()

star_rating
5    1584192
4     418694
1     307234
3     193818
2     138496
Name: count, dtype: int64

 ## Relabeling and Sampling
 
First form three classes and print their statistics. Then randomly select 100,000 reviews from the positive and 100,000 reviews from the negative



In [7]:
def transform(row):
    rating = row['star_rating']
    if rating < 3:
        return 0
    elif rating > 3:
        return 1
    else:
        return 0.5

data['sentiment'] = data.apply(transform, axis=1)

In [23]:
print(f"Positive reviews: {len(data[data['sentiment'] == 1])}")
print(f"Negative reviews: {len(data[data['sentiment'] == 0])}")
print(f"Neutral reviews: {len(data[data['sentiment'] == 0.5])}")

Positive reviews: 2002886
Negative reviews: 445730
Neutral reviews: 193818


In [9]:
positive_reviews = data[data['sentiment'] == 1].sample(100000, random_state=42)
negative_reviews = data[data['sentiment'] == 0].sample(100000, random_state=42)
data_downsized = pd.concat([positive_reviews, negative_reviews])

# Data Cleaning



In [10]:
def clean_review(row):
    review = str(row['review_body'])
    text = review.lower()
    text = BeautifulSoup(text, "html.parser").get_text(strip=True)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = contractions.fix(text)
    return text

data_downsized['review'] = data_downsized.apply(clean_review, axis=1)

In [11]:
# Print in py file
data_cleaned = data_downsized.loc[:, ['review', 'sentiment']]
print(f"Average length before cleaning: {data_downsized['review_body'].astype(str).str.len().mean():.4f}")
print(f"Average length after cleaning: {data_cleaned['review'].apply(len).mean():.4f}")

Average length before cleaning: 318.6052
Average length after cleaning: 302.2660


# Pre-processing

## remove the stop words 

In [12]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def remove_stopwords(row):
    text = row['review']
    words = word_tokenize(text)
    stop = set(stopwords.words('english'))
    return " ".join([w for w in words if w not in stop])

data_cleaned['review'] = data_cleaned.apply(remove_stopwords, axis=1)

## perform lemmatization  

In [13]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize(row):
    text = row['review']
    words = word_tokenize(text)
    return " ".join([lemmatizer.lemmatize(w) for w in words])

data_cleaned['review'] = data_cleaned.apply(lemmatize, axis=1)

In [14]:
showcase = pd.DataFrame()
showcase['pre-cleaning'] = data_downsized.head(3)['review_body']
showcase['post-processing'] = data_cleaned.head(3)['review']
showcase

,pre-cleaning,post-processing
1707224,I really like this organizer. It has all the ...,really like organizer right size compartment t...
1205552,"Binders like this got me through high school, ...",binder like got high school continue useful ye...
455154,Good quality except the metal bar is too wide ...,good quality except metal bar wide hangar


In [15]:
print(f"Average length before preprocessing: {data_downsized['review_body'].astype(str).str.len().mean():.4f}")
print(f"Average length after preprocessing: {data_cleaned['review'].apply(len).mean():.4f}")

Average length before preprocessing: 318.6052
Average length after preprocessing: 187.9352


# Bigram Feature Extraction

In [16]:
from nltk.util import bigrams

def extract_features(row):
    text = row['review']
    tokens = word_tokenize(text)
    pairs = list(bigrams(tokens))
    return " ".join([f"{x}_{y}" for x, y in pairs])

dataset = data_cleaned.loc[:, ['sentiment']]
dataset['review'] = data_cleaned.apply(extract_features, axis=1)

Vectorize text, and split training and test data

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

vec = TfidfVectorizer()

X = vec.fit_transform(dataset['review'])
y = dataset['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Perceptron

In [18]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def test_model(model, name):
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_pred = model.predict(X_test)
    train_precision, train_recall, train_f1, train_support = precision_recall_fscore_support(y_train, y_train_pred, average="binary")
    test_precision, test_recall, test_f1, test_support = precision_recall_fscore_support(y_test, y_pred, average="binary")
    print(f"{name} Train Accuracy: {accuracy_score(y_train, y_train_pred):.4f}")
    print(f"{name} Train Precision: {train_precision:.4f}")
    print(f"{name} Train Recall: {train_recall:.4f}")
    print(f"{name} Train F1: {train_f1:.4f}")
    print(f"{name} Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"{name} Test Precision: {test_precision:.4f}")
    print(f"{name} Test Recall: {test_recall:.4f}")
    print(f"{name} Test F1: {test_f1:.4f}")

In [19]:
from sklearn.linear_model import Perceptron
test_model(Perceptron(random_state=42), "Perceptron")

Perceptron Train Accuracy: 0.9897
Perceptron Train Precision: 0.9829
Perceptron Train Recall: 0.9968
Perceptron Train F1: 0.9898
Perceptron Test Accuracy: 0.8517
Perceptron Test Precision: 0.8394
Perceptron Test Recall: 0.8697
Perceptron Test F1: 0.8543


# SVM

In [20]:
from sklearn.svm import LinearSVC
test_model(LinearSVC(random_state=42), "SVM")

SVM Train Accuracy: 0.9906
SVM Train Precision: 0.9828
SVM Train Recall: 0.9987
SVM Train F1: 0.9907
SVM Test Accuracy: 0.8703
SVM Test Precision: 0.8403
SVM Test Recall: 0.9143
SVM Test F1: 0.8757


# Logistic Regression

In [21]:
from sklearn.linear_model import LogisticRegression
test_model(LogisticRegression(random_state=42), "Logistic Regression")

Logistic Regression Train Accuracy: 0.9553
Logistic Regression Train Precision: 0.9488
Logistic Regression Train Recall: 0.9625
Logistic Regression Train F1: 0.9556
Logistic Regression Test Accuracy: 0.8684
Logistic Regression Test Precision: 0.8510
Logistic Regression Test Recall: 0.8931
Logistic Regression Test F1: 0.8716


# Naive Bayes

In [22]:
from sklearn.naive_bayes import MultinomialNB
test_model(MultinomialNB(), "Multinomial Naive Bayes")

Multinomial Naive Bayes Train Accuracy: 0.9642
Multinomial Naive Bayes Train Precision: 0.9667
Multinomial Naive Bayes Train Recall: 0.9614
Multinomial Naive Bayes Train F1: 0.9641
Multinomial Naive Bayes Test Accuracy: 0.8729
Multinomial Naive Bayes Test Precision: 0.8689
Multinomial Naive Bayes Test Recall: 0.8782
Multinomial Naive Bayes Test F1: 0.8735
